Here are the functions we will be using during our model training: 

In [ ]:
##Create a loop to add the txt files into a data frame
def txt_retrieval(folder_path):
    qa = []
    txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]
    for file in txt_files:
        file_path = os.path.join(folder_path, file) 
        df = pd.read_json(file_path) 
        df["source_file"] = file  
        qa.append(df)
    return pd.concat(qa, ignore_index=True) if qa else pd.DataFrame()


#Combine questions and answers to pass to the model
def qa_pairs(questions, options):
    pairs = []
    for q, opts in zip(questions, options):
        for opt in opts:
            pairs.append((q,opt))
    return pairs


##Use the tokenizer to encode the text
def encode(data_component):
    for i in data_component: 
        encoded_data = tokenizer(data_component, return_tensors='pt', padding=True)
    return encoded_data







Import the question sets that will be used to train the model.  The first dataset is the RACE dataseet, which consists of multiple choice questions separated between M (middle school) and H (high school)

In [ ]:
import pandas as pd
import os

middle = "middle"
high = "high\high"


# Assign separate outputs based on the variable names
m_qa = txt_retrieval(middle)
h_qa = txt_retrieval(high)
    

Let's take a look at the column names to see how the data is structured. 

In [ ]:
m_qa.columns

In [ ]:
h_qa

Separate the data into its components.

In [ ]:
m_questions = m_qa.questions.values.tolist()
h_questions = h_qa.questions.values.tolist()
m_options = m_qa.options.values.tolist()
h_options = h_qa.options.values.tolist()
m_article = m_qa.article.values.tolist()
h_article = h_qa.article.values.tolist()
m_id = m_qa.id.values.tolist()
h_id = h_qa.id.values.tolist()
m_answers = m_qa.answers.values.tolist()
h_answers = h_qa.answers.values.tolist()

Import Bert from transformers and use the tokenizer specialized for the model to input text as tokens. 

In [ ]:
import torch
from transformers import BertConfig, BertModel, AutoModel, AutoTokenizer
bert = AutoModel.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')  

In [ ]:
      
m_qa_pairs = qa_pairs(m_questions, m_options)
m_qa_pairs

Encode questions, option, and answer components.   Save the encoded QA inputs to avoid the time consumption required from the tokenization.  

In [ ]:
m_qa_inputs = encode(m_qa_pairs)

In [ ]:
torch.save(m_qa_inputs, "m_qa_inputs.pt")

Now do the same for the correct answers:

In [ ]:
m_answers_inputs = encode(m_answers)

In [56]:
torch.save(m_answers_inputs, "m_answers_inputs.pt")

And for the readings:

In [57]:
m_readings_inputs = encode(m_article)


In [58]:
torch.save(m_readings_inputs, "m_readings_inputs.pt")